In [6]:
import numpy as np
import torch
import os
import sys
import yaml
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# helpers_path = os.path.join('/ether/aegis/Research_HEP/NRAD/oldver/NRAD/non-resonant-AD/model_scripts')
helpers_path = os.path.join('/ether/aegis/Research_HEP/NRAD/model_scripts')
sys.path.insert(0, os.path.abspath(helpers_path))
from Classifier import Classifier
# from SimpleMAF import SimpleMAF

In [7]:
seed = 2
data_path = f"SemiVisJets/data/data_seed{seed}"
samples_path = "SemiVisJets/samples"
eval_dir = "SemiVisJets/eval_cr"

In [8]:
CUDA = torch.cuda.is_available()
device = torch.device("cuda" if CUDA else "cpu")
print("Device:", device)

config_path = "oldver/NRAD/non-resonant-AD/configs"
with open(f"{config_path}/bc_discrim.yml", 'r') as stream:
    params = yaml.safe_load(stream)
n_context = 2

Device: cuda


In [9]:
def run_eval(set_1, set_2, code, save_dir, classifier_params, device, w_1 = None, w_2 = None, classifier_runs = 20):
    #Set 1 Samples
    #Set 2 Data
    # They must be the same
    # ---------- Handle Weights ----------
    
    # if w_1 is None or w_1.size == 0:
    #     w_1 = np.ones(set_1.shape[0])
    # if w_2 is None or w_2.size == 0:
    #     w_2 = np.ones(set_2.shape[0])
        
    # # define testsets from the input data
    # num_test = min(10000, set_1.shape[0] // 5)

    # trainset_1, testset_1 = set_1[:-num_test], set_1[-num_test:]
    # trainset_2, testset_2 = set_2[:-num_test], set_2[-num_test:]

    # wtrain_1, wtest_1 = w_1[:-num_test], w_1[-num_test:]
    # wtrain_2, wtest_2 = w_2[:-num_test], w_2[-num_test:]

    # # ---------- Build train/test sets ----------
    # input_x_train = np.concatenate([trainset_1, trainset_2], axis=0)
    # input_y_train = np.concatenate([
    #     np.zeros(trainset_1.shape[0]),
    #     np.ones(trainset_2.shape[0])
    # ], axis=0).reshape(-1, 1)
    
    # input_w_train = np.concatenate([wtrain_1, wtrain_2], axis=0).reshape(-1, 1)


    # input_x_test = np.concatenate([testset_1, testset_2], axis=0)
    # input_y_test = np.concatenate([
    #     np.zeros(testset_1.shape[0]),
    #     np.ones(testset_2.shape[0])
    # ], axis=0).reshape(-1, 1)

    # ensure weight arrays exist
    if w_1 is None or w_1.size == 0:
        w_1 = np.ones(set_1.shape[0])
    if w_2 is None or w_2.size == 0:
        w_2 = np.ones(set_2.shape[0])

    # define test size — roughly 20% or limited to 10,000 samples
    test_size_ratio = min(10000 / set_1.shape[0], 0.2)

    # split each dataset independently
    trainset_1, testset_1, wtrain_1, wtest_1 = train_test_split(
        set_1, w_1, test_size=test_size_ratio, random_state=42
    )
    trainset_2, testset_2, wtrain_2, wtest_2 = train_test_split(
        set_2, w_2, test_size=test_size_ratio, random_state=42
    )

    # ---------- Build train/test sets ----------
    # Combine the two datasets
    input_x_train = np.concatenate([trainset_1, trainset_2], axis=0)
    input_y_train = np.concatenate([
        np.zeros(trainset_1.shape[0]),
        np.ones(trainset_2.shape[0])
    ]).reshape(-1, 1)
    input_w_train = np.concatenate([wtrain_1, wtrain_2], axis=0).reshape(-1, 1)

    input_x_test = np.concatenate([testset_1, testset_2], axis=0)
    input_y_test = np.concatenate([
        np.zeros(testset_1.shape[0]),
        np.ones(testset_2.shape[0])
    ]).reshape(-1, 1)
    # input_w_test = np.concatenate([wtest_1, wtest_2], axis=0).reshape(-1, 1)

    
    # ---------- Logging ----------
    print(f"\nWorking on {code}...")
    print("      X_train, y_train, w_train:", input_x_train.shape, input_y_train.shape, input_w_train.shape)
    print("      X_test, y_test:", input_x_test.shape, input_y_test.shape)
    
    aucs_list = []

    for i in range(int(classifier_runs)):
        
        print(f"Classifier run {i+1} of {classifier_runs}.")
        local_id = f"{code}_run{i}"
                
        # train classifier
        NN = Classifier(n_inputs=5, layers=classifier_params["layers"], learning_rate=classifier_params["learning_rate"], device=device, scale_data=False)
        print("Using device:", NN.device)
        NN.train(input_x_train, input_y_train, weights=input_w_train,  save_model=True, model_name = f"model_{local_id}" , n_epochs=classifier_params["n_epochs"], seed = i, outdir=save_dir)

        scores = NN.evaluation(input_x_test)
        auc = roc_auc_score(input_y_test, scores, sample_weight=np.concatenate([wtest_1, wtest_2]))
        if auc < 0.5:
            auc = 1.0 - auc  # symmetry adjustment
        aucs_list.append(auc)
        print(f"   AUC: {auc}")
    
    # ---------- Save results ----------
    os.makedirs(f"{save_dir}/auc_scores", exist_ok=True)
    np.savez(f"{save_dir}/auc_scores/auc_{code}.npz", auc_scores=np.array(aucs_list))

    print("\nMedian AUC, 16th percentile, 84th percentile:")
    print(np.median(aucs_list), [np.percentile(aucs_list, 16), np.percentile(aucs_list, 84)])
    print("Done.\n")


In [ ]:
print("CWoLA Evaluation on Reweight Samples")
for i in range(1, 11):
    reweight_events = np.load(f"{samples_path}/reweight_MC{seed:02d}_Data{i:02d}_CR_samples.npz", allow_pickle=True)
    data_events = np.load(f"{data_path}/data_events_chunk{i:02d}.npz", allow_pickle=True)
    set_1 = reweight_events["mc_cr"][:, n_context:]
    w_1 = reweight_events["w_cr"]
    set_2 = data_events["data_events_cr"][:, n_context:]
    run_eval(set_1, set_2, w_1 = w_1, code=f"reweight_MC{seed:02d}_Data{i:02d}_cr", save_dir=eval_dir, classifier_params=params, device=device)


Working on reweight_MC02_Data01_cr...
      X_train, y_train, w_train: (19916593, 5) (19916593, 1) (19916593, 1)
      X_test, y_test: (20000, 5) (20000, 1)
Classifier run 1 of 20.
Using device: cuda


 20%|==        | 10/50 [15:16<1:01:05, 91.63s/it]


   AUC: 0.5075131822471484
Classifier run 2 of 20.
Using device: cuda


 16%|=>        | 8/50 [13:13<1:09:25, 99.18s/it]


   AUC: 0.5104444106804659
Classifier run 3 of 20.
Using device: cuda


...

   AUC: 0.5114092335763829
Classifier run 19 of 20.
Using device: cuda


 18%|=>        | 9/50 [17:56<1:21:42, 119.57s/it]


   AUC: 0.5113825412926665
Classifier run 20 of 20.
Using device: cuda


 20%|==        | 10/50 [19:24<1:17:38, 116.46s/it]


   AUC: 0.5096525141587295

Median AUC, 16th percentile, 84th percentile:
0.5108675335508218 [0.5094523156498598, 0.5114614915809681]
Done.



In [ ]:
for i in range(1, 11):
    generate_events = np.load(f"{samples_path}/generate_MC{seed:02d}_Data{i:02d}_CR_samples.npz", allow_pickle=True)
    context_weights = np.load(f"{samples_path}/context_weight_MC{seed:02d}_Data{i:02d}_CR_samples.npz", allow_pickle=True)
    data_events = np.load(f"{data_path}/data_events_chunk{i:02d}.npz", allow_pickle=True)
    set_1 = generate_events["generate_cr"]
    set_2 = data_events["data_events_cr"][:, n_context:]
    run_eval(set_1, set_2, code=f"generate_MC{seed:02d}_Data{i:02d}_cr", save_dir=eval_dir, classifier_params=params, device=device)


Working on generate_MC02_Data01_cr...
      X_train, y_train, w_train: (19947488, 5) (19947488, 1) (19947488, 1)
      X_test, y_test: (20000, 5) (20000, 1)
Classifier run 1 of 20.
Using device: cuda


  6%|>         | 3/50 [04:06<1:04:07, 81.86s/it]